In [2]:
import pandas as pd
import numpy as np
import pickle
from lightgbm import LGBMRegressor
MODELS_CACHE = {}

In [3]:
df = pd.read_csv("full_data.csv")

In [4]:
pitchers = list(df["pitcherId"].unique())
pitches = list(df["pitchType"].unique())
pitches.remove("KN")
pitches.remove("ST")
lr = ["L", "R"]

features = ['inducedVertBreak', 'horzBreak', 'extension', 'relX', 'relZ', 'releaseVelocity', 'spinRate']
targets = ["rv"]

test = df[df["year"] == 2026]

In [5]:
def pred_rv(test, features, target, pitch, side):
    if (pitch, side) not in MODELS_CACHE:
        with open(f'models/26_{pitch}_{side}.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, side)] = pickle.load(f)

    model = MODELS_CACHE[(pitch, side)]
    X_test = test[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [6]:
dfs = []

for pitcher in pitchers: 
    for pitch in pitches: 
        for side in lr: 
            te = test[(test["pitcherId"] == pitcher) & (test["pitchType"] == pitch) & (test["pitcherHand"] == side)] 
            if len(te) >= 10: 
                dfx = {} 
                dfx["Pitcher"] = te["Pitcher"].iloc[0]
                dfx["pitcherId"] = pitcher
                dfx["Team"] = te["pitchingTeam"].iloc[0]
                dfx["PitchType"] = pitch 
                dfx["PitcherThrows"] = side 
                dfx["Count"] = len(te) 
                for feature in features: 
                    dfx[feature] = te[feature].mean() 
                for target in targets: 
                    dfz = pd.DataFrame([dfx], columns=features) 
                    dfx["xrv"] = float(pred_rv(dfz, features, target, pitch, side).iloc[0]) 
                dfs.append(dfx)

In [7]:
stuff = pd.DataFrame(dfs).reset_index(drop=True)

In [8]:
stuff[stuff["Team"] == "URI"].sort_values("xrv")

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv
2836,E. Maloney,1085415168,URI,CH,R,29,9.200727,11.137915,4.980640,-0.189884,6.829958,76.225244,1451.312982,0.020714
2846,J. Kopetski,1269533184,URI,FC,L,19,6.771141,4.331412,6.235199,2.088972,5.640634,75.404928,1738.530242,0.045295
3409,J. Sabbath,1110853376,URI,SL,R,160,5.175643,-11.907250,5.781683,-1.805022,5.680606,78.781352,2350.915489,0.050557
10377,L. Lavigueur,1143021056,URI,SL,R,76,-5.820478,-20.799608,5.245887,-1.487878,5.928555,78.427225,2515.432751,0.055412
3392,J. Cullen,1358089441,URI,FA,R,203,22.665115,10.827551,5.994514,-1.004210,6.129405,91.638213,2253.720642,0.055759
3390,D. Asencio,1241946368,URI,SL,R,12,6.660355,-11.770542,5.675755,-1.935682,5.841291,75.512682,2326.185922,0.060447
3986,C. Grotyohann,1203468032,URI,FA,R,38,22.637336,8.417300,5.913379,-1.917919,6.220504,90.554157,2188.620738,0.063762
3393,J. Cullen,1358089441,URI,SL,R,57,4.681646,-4.079590,5.550405,-1.438953,5.851182,83.456744,2520.911163,0.065969
3406,A. Jones,1101042688,URI,FC,R,14,5.622384,-0.570624,5.430084,-1.466703,5.832419,82.295189,2278.644448,0.067727
2839,C. Maloney,1336247552,URI,CH,R,11,11.786229,11.637562,5.035975,-0.659251,6.603807,77.606979,1253.749276,0.068884


In [9]:
from scipy import stats

#df['zscore'] = stats.zscore(df['column'])
stuff["Stuff+"] = 100 + (-10 * stuff.groupby("PitchType")["xrv"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [10]:
stuff.sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
6175,C. Oliff,1579384799,ETSU,SL,R,54,11.776464,-5.543496,6.070116,-5.504503,3.394192,76.588125,2223.249877,-0.026250,158
4428,L. Earnhardt,1160764160,WIN,FS,L,66,1.416358,-6.970572,5.306394,1.314118,6.493396,78.006762,1104.887054,-0.404758,153
8788,I. Lohse,1102217728,MIZZ,FC,L,30,8.650791,0.240221,5.280999,1.145014,4.968095,87.251003,2562.277705,-0.115568,149
5469,D. Volantis,1979617275,TEX,SL,L,20,-0.147991,-1.393318,4.832832,2.250061,6.857666,87.127748,2568.711395,-0.005778,146
2210,O. Pudvar,1909941695,CONN,FC,L,46,10.731068,-0.038378,5.131493,0.902312,5.244803,84.537952,2155.593190,-0.100609,145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6717,R. Haider,1130936576,WSU,SL,R,15,3.280341,-8.424923,5.725601,-4.629531,3.492356,76.656960,2314.115270,0.173495,47
11215,J. Easterlin,1327301888,OAK,CU,R,12,1.393492,-2.239383,6.471502,-0.702690,6.456589,69.045468,1729.885104,0.211094,47
10877,C. Wade,1238048512,ALCN,FA,L,58,16.063925,-2.296364,5.617000,1.627021,5.399237,64.696430,1497.050535,0.245445,46
11604,S. Cooper,1578909932,ULM,SL,R,28,1.885338,-12.742766,5.087992,-1.748968,5.325088,69.337926,1927.949200,0.176279,45


In [11]:
stuff.to_csv("stuff.csv")

In [12]:
stuff[stuff["Count"] > 200].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
6764,S. Matson,1198395392,USC,FA,L,244,16.797915,-16.320475,6.147980,1.708561,4.783717,90.319948,2314.911548,0.030045,136
7406,H. Dietz,1301090561,ARK,FA,L,277,17.110409,-6.439685,6.067808,2.037489,6.813163,95.028413,2394.163303,0.038339,132
2225,C. West,1310862081,CONN,FA,L,237,23.930800,-5.804477,6.038546,0.326793,6.154180,90.451656,2224.082182,0.040783,131
6784,J. Harper,1700152819,CSF,FA,L,297,18.059018,-16.164177,5.913329,1.642396,5.637875,90.162148,2239.679651,0.047020,129
6641,M. Brassfield,1038792681,TCU,SL,L,270,-0.331374,7.461935,5.443446,1.941769,6.261726,84.944340,2747.018745,0.025619,129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5960,M. Eagen,1995173029,QUC,FA,R,236,14.921013,14.029757,6.089453,-1.136470,6.032340,89.128035,2376.843993,0.152633,85
14714,J. Price,1342170246,EKY,FA,R,309,12.780923,12.355080,5.938767,-1.473288,5.870021,86.937260,2320.744913,0.151117,85
14182,C. McMullen,1775801108,KENN,SI,R,213,9.898919,13.015634,5.948060,-1.197313,6.197967,92.420368,1961.651332,0.180219,81
3071,L. Jones,1997059014,CCAR,SI,R,233,8.919814,17.407073,5.573061,-1.604692,5.181255,88.874009,2364.260916,0.183724,79


In [13]:
stuff[(stuff["Team"] == "URI") & (stuff["Count"] >= 11)].sort_values("Stuff+", ascending=False).head(60)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
2836,E. Maloney,1085415168,URI,CH,R,29,9.200727,11.137915,4.980640,-0.189884,6.829958,76.225244,1451.312982,0.020714,133
3392,J. Cullen,1358089441,URI,FA,R,203,22.665115,10.827551,5.994514,-1.004210,6.129405,91.638213,2253.720642,0.055759,125
3986,C. Grotyohann,1203468032,URI,FA,R,38,22.637336,8.417300,5.913379,-1.917919,6.220504,90.554157,2188.620738,0.063762,122
3409,J. Sabbath,1110853376,URI,SL,R,160,5.175643,-11.907250,5.781683,-1.805022,5.680606,78.781352,2350.915489,0.050557,115
10377,L. Lavigueur,1143021056,URI,SL,R,76,-5.820478,-20.799608,5.245887,-1.487878,5.928555,78.427225,2515.432751,0.055412,112
15542,W. Creed,1299086745,URI,FA,L,28,21.377306,-14.409316,5.577961,0.522100,6.484818,86.696848,2159.352794,0.086662,112
2839,C. Maloney,1336247552,URI,CH,R,11,11.786229,11.637562,5.035975,-0.659251,6.603807,77.606979,1253.749276,0.068884,112
3400,M. Santos,1487988943,URI,FA,R,108,16.154692,8.102906,5.610465,-2.416509,5.292402,92.753839,2450.193301,0.091720,110
3403,A. Jones,1101042688,URI,FA,R,70,20.956155,8.634500,5.932586,-1.227173,5.846486,89.160199,2129.800095,0.092568,110
2847,C. Johnston,1241946624,URI,CH,R,83,10.891403,13.513926,6.435540,-0.684517,6.099914,79.698941,1724.467899,0.072624,110


In [14]:
stuff[(stuff["Count"] > 50) & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
441,J. Music,1161348114,CAMP,FA,R,112,21.717838,14.051507,5.627380,-3.628881,5.898753,91.553440,2226.384722,0.010936,144
6764,S. Matson,1198395392,USC,FA,L,244,16.797915,-16.320475,6.147980,1.708561,4.783717,90.319948,2314.911548,0.030045,136
9372,S. Garcia,1602557566,LSU,FA,L,118,21.774008,-10.973115,6.184411,1.553425,6.258128,92.701719,2461.242677,0.031237,135
7406,H. Dietz,1301090561,ARK,FA,L,277,17.110409,-6.439685,6.067808,2.037489,6.813163,95.028413,2394.163303,0.038339,132
13954,J. Nottingham,1557710177,UGA,FA,R,97,21.129491,11.899358,6.104974,-1.516178,6.158468,96.271623,2559.413119,0.038118,132
2225,C. West,1310862081,CONN,FA,L,237,23.930800,-5.804477,6.038546,0.326793,6.154180,90.451656,2224.082182,0.040783,131
5502,T. Burns,1207858176,TEX,FA,R,112,23.144534,4.586872,6.105255,-0.695601,6.561737,96.352919,2234.905418,0.042177,131
11644,C. Clark,1956935142,USM,FA,R,177,21.703584,9.329253,6.563693,-1.717891,5.478274,93.931105,2258.346428,0.042846,130
1860,D. Sheerin,1213263616,LSU,FA,R,172,14.680648,15.875060,6.066550,-3.455965,5.459771,96.465379,2442.404512,0.044374,130
5917,J. Soucie,1221790928,UK,FA,L,91,17.319146,-15.088271,5.826764,3.052747,6.390206,92.112573,2208.649834,0.043308,130


In [15]:
stuff[stuff["pitcherId"] == 1110853376]["Count"].sum()

np.int64(520)

In [16]:
import pandas as pd
import numpy as np
stuff = pd.read_csv("stuff.csv")
pitches = list(stuff["PitchType"].unique())

In [17]:
rows = []
ids = list(stuff["pitcherId"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["Team"] = x["Team"].iloc[0]
    dx["PitcherThrows"] = x["PitcherThrows"].iloc[0]
    for pt in list(stuff["PitchType"].unique()):
        if pt in list(x["PitchType"].unique()):
            dx["Stf+ " + pt] = x[x["PitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["PitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)

In [18]:
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

table["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table.columns and ct_col in table.columns:
        mask = table[col].notna()
        table.loc[mask, "Stuff+"] += (
            table.loc[mask, col] * (table.loc[mask, ct_col] / table.loc[mask, "Total"])
        )

table["Stuff+"] = table["Stuff+"].round().astype(int)
table = table[["Pitcher", "pitcherId", "Team", "Total", "PitcherThrows", "Stf+ FA", "Stf+ SL", "Stf+ CU", "Stf+ FC", "Stf+ SI", "Stf+ CH", "Stf+ FS", "Stuff+"]]

In [19]:
table[table["Team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
969,J. Cullen,1358089441,URI,392,R,125,106,103,101,<NA>,96,<NA>,114
1143,C. Grotyohann,1203468032,URI,57,R,122,<NA>,<NA>,96,<NA>,<NA>,<NA>,113
4696,W. Creed,1299086745,URI,28,L,112,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,112
973,J. Sabbath,1110853376,URI,520,R,108,115,100,<NA>,<NA>,103,<NA>,109
971,M. Santos,1487988943,URI,189,R,110,101,<NA>,<NA>,<NA>,<NA>,<NA>,106
796,C. Maloney,1336247552,URI,146,R,107,103,100,<NA>,<NA>,112,<NA>,105
972,A. Jones,1101042688,URI,131,R,110,101,95,103,<NA>,96,<NA>,105
795,E. Maloney,1085415168,URI,128,R,92,<NA>,97,<NA>,<NA>,133,<NA>,102
2993,L. Lavigueur,1143021056,URI,156,R,91,112,<NA>,<NA>,<NA>,<NA>,<NA>,101
968,D. Asencio,1241946368,URI,47,R,96,109,<NA>,<NA>,<NA>,<NA>,<NA>,99


In [28]:
table[table["pitcherId"] == 1307169536]

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+


In [20]:
table[table["Total"] >= 500].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
2137,H. Dietz,1301090561,ARK,645,L,132,104,145,139,<NA>,<NA>,<NA>,130
1575,D. Volantis,1979617275,TEX,548,L,114,146,131,126,125,101,<NA>,125
2054,C. Carlon,1279321088,ASU,671,L,127,123,107,81,<NA>,93,<NA>,121
1369,L. Peterson,1142994176,FLA,690,R,123,123,110,<NA>,<NA>,108,<NA>,120
4298,M. Edwards,1279382784,USC,707,L,129,116,110,<NA>,<NA>,101,<NA>,119
2576,E. Nachtsheim,1256101932,MCNS,655,R,124,110,104,108,<NA>,100,<NA>,119
655,R. Marohn,1197348608,NCST,602,L,118,112,110,<NA>,115,129,<NA>,119
1332,E. Lund,1985708358,OKST,700,L,119,112,120,<NA>,<NA>,<NA>,<NA>,118
2697,W. Schmidt,1280627385,LSU,662,R,115,121,117,<NA>,<NA>,<NA>,<NA>,118
4247,K. Sweum,1558099259,GONZ,537,L,120,125,<NA>,90,<NA>,79,113,117


In [21]:
table[table["Team"] == "MRMK"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
3054,A. Holmes,1425232427,MRMK,34,R,112,107,<NA>,<NA>,<NA>,<NA>,<NA>,110
497,S. Franco,1769668118,MRMK,66,L,91,<NA>,101,<NA>,<NA>,111,<NA>,99
3055,C. Barker,1304701440,MRMK,47,R,99,97,<NA>,<NA>,<NA>,<NA>,<NA>,98
496,C. Kiesiner,1402009816,MRMK,99,R,96,97,<NA>,<NA>,<NA>,<NA>,<NA>,96
498,Z. Broderick,1327271424,MRMK,120,L,95,99,<NA>,<NA>,<NA>,97,<NA>,96
1820,N. Hunkele,1342167952,MRMK,73,R,95,86,<NA>,<NA>,<NA>,107,<NA>,96
634,J. Borsari,1213486080,MRMK,48,R,91,92,<NA>,104,<NA>,<NA>,<NA>,95
636,O. Barkal,1384801355,MRMK,108,R,86,95,<NA>,<NA>,<NA>,<NA>,<NA>,93
1821,J. Nichols,1907374041,MRMK,91,R,86,99,<NA>,99,<NA>,<NA>,<NA>,93
3038,B. Loper,1307046400,MRMK,58,R,96,83,<NA>,<NA>,<NA>,102,<NA>,92


In [31]:
stuff[stuff["pitcherId"] == 1304701440]

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,vertApprAngle,inducedVertBreak,horzBreak,horzApprAngle,extension,relX,relZ,releaseVelocity,spinRate,spinDir,xrv,Stuff+
10561,C. Barker,1304701440,MRMK,FA,R,20,-5.723808,18.083514,13.476466,-0.177295,6.136091,-1.525991,6.408032,89.168222,2222.320350,219.664934,0.107134,95
10562,C. Barker,1304701440,MRMK,SL,R,27,-8.636924,-2.136478,-6.163696,-2.312542,5.749747,-1.587969,6.323121,77.669330,2216.939097,57.785514,0.083488,85


In [22]:
table.to_csv("stuff_table.csv")

In [33]:
table[table["Team"] == "CCAR"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
675,C. Flukey,1238148096,CCAR,59,R,126,<NA>,108,<NA>,<NA>,<NA>,<NA>,117
184,D. Parker,1676824501,CCAR,303,R,117,<NA>,102,<NA>,<NA>,100,<NA>,113
2975,R. Norman,1105483008,CCAR,481,R,104,107,<NA>,<NA>,<NA>,94,<NA>,105
55,J. Appelman,1954026382,CCAR,321,R,92,110,<NA>,<NA>,100,101,<NA>,104
58,S. Doran,1253195181,CCAR,381,R,102,103,107,<NA>,<NA>,<NA>,<NA>,104
78,C. Bosch,1750954648,CCAR,390,L,100,113,<NA>,<NA>,<NA>,108,<NA>,104
54,C. Kane,1162293760,CCAR,219,R,102,98,104,108,<NA>,<NA>,<NA>,103
53,J. Smallets,1317717863,CCAR,216,R,97,106,<NA>,<NA>,101,<NA>,<NA>,102
56,R. Lynch,1185766656,CCAR,92,R,106,99,103,<NA>,<NA>,<NA>,<NA>,102
57,D. Horn,1297495808,CCAR,545,R,98,98,120,86,104,<NA>,<NA>,99
